# CardioTwin Hybrid XGBoost–LSTM Evaluation Report

Evaluates the 30-day MACE prediction pipeline on a held-out 20% test cohort (`n = 900` longitudinal patient trajectories) against:
- **Static Baseline Calculator** (Framingham/ASCVD-style point-in-time logistic score)
- **XGBoost Branch** (26 tabular + derived hemodynamic features)
- **PyTorch LSTM Branch** (12-step longitudinal vitals/biomarker sequences)
- **Hybrid XGBoost–LSTM Meta-Learner** (calibrated late fusion)

Reports **ROC-AUC**, **Calibration (Brier Score & Reliability Curve)**, and **False-Positive Rate at Matched 80% Sensitivity**.

In [ ]:
import json
from pathlib import Path
import numpy as np
import joblib
import torch
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from backend.models.train import generate_synthetic_cohort, ARTIFACTS_DIR
from backend.models.features import TS_FEATURE_NAMES
from backend.models.lstm_net import VitalsLSTM

In [ ]:
X_tab, X_seq, y, p_static = generate_synthetic_cohort(n_patients=4500, seed=42)
indices = np.arange(len(y))
_, idx_test = train_test_split(indices, test_size=0.20, stratify=y, random_state=42)

xgb = XGBClassifier()
xgb.load_model(str(ARTIFACTS_DIR / 'xgb_model.json'))
seq_scaler = joblib.load(ARTIFACTS_DIR / 'seq_scaler.joblib')
meta = joblib.load(ARTIFACTS_DIR / 'meta_model.joblib')

lstm = VitalsLSTM(input_size=len(TS_FEATURE_NAMES), hidden_size=48, num_layers=2, dropout=0.2)
lstm.load_state_dict(torch.load(ARTIFACTS_DIR / 'lstm_model.pt', map_location='cpu', weights_only=True))
lstm.eval()

p_xgb = xgb.predict_proba(X_tab[idx_test])[:, 1]
n_te, T, F = X_seq[idx_test].shape
X_seq_sc = seq_scaler.transform(X_seq[idx_test].reshape(-1, F)).reshape(n_te, T, F)
with torch.no_grad():
    p_lstm = torch.sigmoid(lstm(torch.tensor(X_seq_sc, dtype=torch.float32))).numpy()

eps = 1e-5
px = np.clip(p_xgb, eps, 1 - eps)
pl = np.clip(p_lstm, eps, 1 - eps)
meta_x = np.column_stack([np.log(px / (1 - px)), np.log(pl / (1 - pl)), px * pl])
p_hybrid = meta.predict_proba(meta_x)[:, 1]
y_test = y[idx_test]
p_base = p_static[idx_test]

with open(ARTIFACTS_DIR / 'evaluation_metrics.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curves
for name, scores in [
    ('Static Baseline', p_base),
    ('XGBoost Only', p_xgb),
    ('LSTM Only', p_lstm),
    ('Hybrid XGBoost-LSTM', p_hybrid),
]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].axhline(0.80, color='gray', linestyle=':', label='80% Sensitivity Target')
axes[0].set_title('30-Day MACE ROC Comparison')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Sensitivity)')
axes[0].legend(loc='lower right')

# Calibration Reliability Curve
for name, scores in [('Static Baseline', p_base), ('Hybrid XGBoost-LSTM', p_hybrid)]:
    prob_true, prob_pred = calibration_curve(y_test, scores, n_bins=10)
    brier = brier_score_loss(y_test, scores)
    axes[1].plot(prob_pred, prob_true, marker='o', label=f'{name} (Brier = {brier:.4f})')

axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfectly Calibrated')
axes[1].set_title('Probability Calibration Curve')
axes[1].set_xlabel('Predicted 30-Day MACE Probability')
axes[1].set_ylabel('Observed Event Frequency')
axes[1].legend(loc='upper left')

plt.tight_layout()
plt.show()